# KLOT Quasi-Vertical Profile — 29–30 August 2022 Convective Event

A time-height **quasi-vertical profile (QVP)** of a nocturnal convective event over
Chicagoland, paired frame-by-frame with the low-level reflectivity PPI, read straight
from the analysis-ready **NEXRAD ARCO** Icechunk/Zarr store on AWS
(`s3://nexrad-arco`, https://registry.opendata.aws/nexrad-arco/).

**What this notebook does**

1. Opens the KLOT (Chicago WSR-88D) repo in the ARCO store — anonymously, no AWS credentials.
2. Builds a QVP by azimuthally averaging the 4.0° tilt (`sweep_9`, VCP-212) over a ~5.3 h
   window, with a ρHV > 0.80 quality mask so noise gates aloft do not contaminate the mean.
3. **Stitches** the QVP onto an evenly-spaced scan-index axis, dropping the scans where the
   4° cut was left unpopulated — removing the white vertical bars a true-time axis would show.
4. Renders a combined animation: the 0.5° reflectivity PPI on the left, the stitched
   three-panel QVP (Z, Z_DR, ρHV) on the right with a moving time cursor.

All colormaps are colorblind-friendly (`ChaseSpectral`, `balance`, `plasma`).

> **Melting/QC note.** VCP-212 leaves the upper tilts empty at many archived times, so the 4°
> cut has genuine data gaps; those scans are dropped rather than shown as blank columns.
> Five scans in this window carry a corrupt time-coordinate unit that breaks Py-ART's sweep
> stacking and are skipped in the animation loop.


## 1. Setup

The store's chunks are read through the `rustytree` engine (`pip install rustytree-xarray`).
icechunk's native (Rust) S3 client does not read Python's `certifi` bundle by default and
fails TLS with `UnknownIssuer`, so we point the standard CA environment variables at certifi
**before** the first `import icechunk`.

In [ ]:
import os, copy, gc
import numpy as np
import pandas as pd
import xarray as xr
from xarray import DataTree

# CA trust for icechunk's native S3 client — must precede `import icechunk`
import certifi
_CA = certifi.where()
for _v in ("SSL_CERT_FILE", "AWS_CA_BUNDLE", "REQUESTS_CA_BUNDLE", "CURL_CA_BUNDLE"):
    os.environ.setdefault(_v, _CA)

import icechunk
import xradar          # registers the .xradar accessor
import pyart
import cmweather       # registers ChaseSpectral, balance, etc.

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs

BUCKET = "nexrad-arco"
ENGINE = "rustytree"
RADAR  = "KLOT"        # Chicago WSR-88D: 41.604 N, -88.084 W, 231 m
KLOT   = (41.60444, -88.08444)

## 2. Connect to the ARCO store

The store is one Icechunk repo per radar, grouped by VCP (volume coverage pattern) then
sweep. Each `/VCP-xxx/sweep_N` is a single time-stacked cube with a `vcp_time` axis spanning
2015 → near-real-time.

In [ ]:
def connect(prefix, branch="main", region="us-east-1"):
    """Open a read-only, anonymous Icechunk session on s3://nexrad-arco/<prefix>."""
    storage = icechunk.s3_storage(bucket=BUCKET, prefix=prefix, region=region,
                                  anonymous=True)
    virtual_auth = icechunk.containers_credentials(
        {"s3://%s/" % BUCKET: icechunk.s3_anonymous_credentials()})
    repo = icechunk.Repository.open(storage,
                                    authorize_virtual_chunk_access=virtual_auth)
    return repo.readonly_session(branch)

session = connect(RADAR)
VCP = "VCP-212"                       # this event ran the dual-pol precip VCP
print("connected to", RADAR)

## 3. Read the 4.0° tilt and build the QVP

A QVP is formed by averaging a single higher tilt over all azimuths, turning each volume scan
into one vertical column. For VCP-212, `sweep_9` (nominal 4.0°, here 3.955°) is the highest
tilt reliably populated in this archive segment. Beam height uses the 4/3-Earth-radius
approximation:

$$h(r) = \sqrt{r^2 + R_e^2 + 2\,r\,R_e\sin(\theta_e)} - R_e, \qquad R_e = \tfrac{4}{3}\times 6{,}371{,}000\ \mathrm{m}$$

**Quality control:** a gate contributes to the azimuthal mean only where ρHV > 0.80 and the
gate is finite; each height bin further requires ≥ 15 valid azimuths. Without this, the
un-thresholded ρHV mean aloft collapses to the noise floor (~0.7) instead of the true
meteorological ~0.99.

In [ ]:
# --- read sweep_9 as a time cube (lazy) ---
n9 = xr.open_datatree(session.store, engine=ENGINE,
                      group_filter="/%s/sweep_9" % VCP)["%s/sweep_9" % VCP]
elev = float(np.nanmedian(n9["sweep_fixed_angle"].values))
rng  = n9["range"].values                      # metres, 250 m gate spacing
print("sweep_9 elevation %.3f deg, %d range gates to %.0f km"
      % (elev, rng.size, rng[-1] / 1e3))

# --- beam height (4/3 earth) ---
Re = 4.0 / 3.0 * 6_371_000.0
h_m = np.sqrt(rng**2 + Re**2 + 2 * rng * Re * np.sin(np.deg2rad(elev))) - Re
h_km = h_m / 1e3

In [ ]:
# --- convective window: 2022-08-29 ~21:53 UTC -> 2022-08-30 ~03:16 UTC ---
vt = pd.to_datetime(n9["vcp_time"].values)
i0 = int(np.searchsorted(vt, pd.Timestamp("2022-08-29 21:50")))
i1 = int(np.searchsorted(vt, pd.Timestamp("2022-08-30 03:17")))
print("window indices %d..%d  (%s -> %s)" % (i0, i1, vt[i0], vt[i1 - 1]))

def azavg_qc(i, rho_min=0.80, min_az=15):
    """Azimuthal mean of DBZH/ZDR/RHOHV at time index i, masked to rho>rho_min."""
    dbz = n9["DBZH"].isel(vcp_time=i).values.astype("f4")
    zdr = n9["ZDR"].isel(vcp_time=i).values.astype("f4")
    rho = n9["RHOHV"].isel(vcp_time=i).values.astype("f4")
    m = np.isfinite(rho) & (rho > rho_min) & np.isfinite(dbz)
    ok = m.sum(axis=0) >= min_az
    def avg(a):
        with np.errstate(invalid="ignore"):
            v = np.nanmean(np.where(m, a, np.nan), axis=0)
        return np.where(ok, v, np.nan)
    return avg(dbz), avg(zdr), avg(rho)

idx = np.arange(i0, i1)
Z  = np.full((idx.size, rng.size), np.nan, "f4")
ZD = np.full_like(Z, np.nan)
RH = np.full_like(Z, np.nan)
for k, i in enumerate(idx):
    Z[k], ZD[k], RH[k] = azavg_qc(int(i))
times = vt[i0:i1]
print("raw QVP:", Z.shape, "  RHOHV mean 3-6 km = %.3f"
      % np.nanmean(RH[:, (h_km > 3) & (h_km < 6)]))

## 4. Stitch: drop empty scans, plot on an even scan-index axis

Plotting on the true time axis renders the unpopulated scans as blank vertical bars. Instead
we keep only columns with ≥ 5 % valid gates (the populated 4° profiles) and lay them on an
evenly-spaced index axis with abutting equal-width columns. Time tick labels are mapped back
to the retained column positions.

In [ ]:
# keep populated columns (>=5% valid gates in the 0-10 km cap)
cap = h_km <= 10.0
valid = np.isfinite(Z[:, cap]).mean(axis=1) >= 0.05
Zs, ZDs, RHs = Z[valid][:, cap], ZD[valid][:, cap], RH[valid][:, cap]
tv = times[valid]
hk = h_km[cap]
nX = tv.size
print("kept %d of %d columns (dropped %d empty)  span %s -> %s"
      % (nX, valid.size, (~valid).sum(), tv[0], tv[-1]))

# cell edges for pcolormesh (even index axis + height)
xe = np.arange(nX + 1)
dh = np.diff(hk); he = np.concatenate([[hk[0] - dh[0] / 2],
                                       hk[:-1] + dh / 2, [hk[-1] + dh[-1] / 2]])

# ~7 evenly spaced time ticks mapped to kept-column positions
tick_i = np.linspace(0, nX - 1, 7).round().astype(int)
tick_lab = [pd.Timestamp(tv[i]).strftime("%H:%M") for i in tick_i]

QV = dict(Z=Zs, ZD=ZDs, RH=RHs, xe=xe, he=he, hk=hk,
          tick_i=tick_i, tick_lab=tick_lab, nX=nX)
qpanels = [("Z",  "ChaseSpectral", 0,  55, r"$Z_H$ (dBZ)"),
           ("ZD", "balance",       -1,  4, r"$Z_{DR}$ (dB)"),
           ("RH", "plasma",       0.9, 1.0, r"$\rho_{HV}$")]

In [ ]:
# standalone stitched QVP
fig = plt.figure(figsize=(11, 8))
gs = gridspec.GridSpec(3, 1, hspace=0.18, left=0.08, right=0.98, top=0.95, bottom=0.08)
for r, (key, cmap, vmn, vmx, cbl) in enumerate(qpanels):
    ax = fig.add_subplot(gs[r])
    pm = ax.pcolormesh(QV["xe"], QV["he"], QV[key].T, cmap=cmap, vmin=vmn, vmax=vmx,
                       shading="flat", rasterized=True)
    cb = fig.colorbar(pm, ax=ax, pad=0.012, fraction=0.046); cb.set_label(cbl)
    ax.set_ylim(0, 10); ax.set_ylabel("Height (km)")
    ax.axhline(4.0, color="0.3", lw=0.8, ls="--")     # ~0 degC level for this case
    ax.set_xlim(0, QV["nX"]); ax.set_xticks(QV["tick_i"] + 0.5)
    ax.set_xticklabels(QV["tick_lab"] if r == 2 else [])
    if r == 0: ax.set_title("KLOT QVP (4.0 deg tilt) — 2022-08-29/30", loc="left")
    if r == 2: ax.set_xlabel("Time (UTC)")
fig.savefig("KLOT_QVP_stitched_20220830.png", dpi=140, bbox_inches="tight")
plt.show()

## 5. Combined PPI + QVP animation

Each frame pairs the 0.5° reflectivity PPI (Py-ART's own gate lat/lon on a Mercator map, no
gatefilter so low-level structure is preserved) with the stitched QVP carrying a moving black
time cursor. The `to_pyart` helper selects one timestamp across the VCP's sweeps, rebuilds a
CfRadial2 DataTree with 0-based sweep numbering (required by Py-ART's Xradar accessor), and
adds standard field-name aliases.

In [ ]:
_ALIAS = {"reflectivity": "DBZH", "cross_correlation_ratio": "RHOHV",
          "differential_reflectivity": "ZDR", "differential_phase": "PHIDP",
          "velocity": "VRADH", "spectrum_width": "WRADH"}
_UNITS = {"reflectivity": "dBZ", "cross_correlation_ratio": "",
          "differential_reflectivity": "dB", "differential_phase": "deg",
          "velocity": "m/s", "spectrum_width": "m/s"}

def _sweep_names(vcp):
    dt = xr.open_datatree(session.store, engine=ENGINE, group_filter="/%s/*" % vcp)
    return sorted((g.split("/")[-1] for g in dt.groups
                   if g.split("/")[-1].startswith("sweep_")),
                  key=lambda s: int(s.split("_")[1]))

def to_pyart(t, nsweeps=1):
    """Py-ART Radar for the volume at time t (nearest), first nsweeps sweeps."""
    t = pd.Timestamp(t)
    names = _sweep_names(VCP)[:nsweeps]
    sweeps, fixed = [], []
    for i, sname in enumerate(names):
        g = "/%s/%s" % (VCP, sname)
        dsk = (xr.open_datatree(session.store, engine=ENGINE, group_filter=g)[g[1:]]
               .to_dataset(inherit="all_coords").sel(vcp_time=t, method="nearest"))
        dsk = dsk.drop_vars([v for v in ["vcp_time"] if v in dsk.coords]).copy()
        dsk["sweep_number"] = xr.DataArray(np.int64(i))       # 0-based, group order
        sweeps.append(dsk); fixed.append(float(dsk.sweep_fixed_angle.values))
    root = xr.Dataset(
        coords=dict(latitude=sweeps[0].latitude, longitude=sweeps[0].longitude,
                    altitude=sweeps[0].altitude),
        data_vars=dict(sweep_fixed_angle=("sweep", np.array(fixed)),
                       sweep_group_name=("sweep",
                           np.array(["sweep_%d" % i for i in range(len(sweeps))]))))
    tree = DataTree(root, children={"sweep_%d" % i: DataTree(d)
                                    for i, d in enumerate(sweeps)})
    tree.attrs["Conventions"] = "Cf/Radial"
    radar = pyart.xradar.Xradar(tree, default_sweep="sweep_0")
    for std, odim in _ALIAS.items():
        if odim in radar.fields and std not in radar.fields:
            radar.fields[std] = copy.deepcopy(radar.fields[odim])
            radar.fields[std]["units"] = _UNITS[std]
    return radar

In [ ]:
EXTENT = [-89.8, -86.4, 40.25, 42.95]     # ~ +/- 150 km around KLOT

def render_frame(fi, t, path):
    fig = plt.figure(figsize=(13.5, 8.2))
    gs = gridspec.GridSpec(3, 2, width_ratios=[1.15, 1.0], hspace=0.28, wspace=0.16,
                           left=0.04, right=0.955, top=0.9, bottom=0.09)
    radar = to_pyart(t, nsweeps=1)

    # --- PPI (Py-ART own gate lat/lon; no gatefilter) ---
    axm = fig.add_subplot(gs[:, 0], projection=ccrs.Mercator())
    axm.set_extent(EXTENT, crs=ccrs.PlateCarree())
    disp = radar.get_gate_lat_lon_alt(0)
    glat, glon = disp[0], disp[1]
    zz = radar.get_field(0, "reflectivity")
    pm = axm.pcolormesh(glon, glat, zz, cmap="ChaseSpectral", vmin=-10, vmax=70,
                        transform=ccrs.PlateCarree(), shading="auto", zorder=5)
    axm.coastlines("10m", lw=0.5)
    import cartopy.feature as cfeature
    axm.add_feature(cfeature.STATES.with_scale("10m"), lw=0.4, edgecolor="0.4")
    axm.plot(KLOT[1], KLOT[0], "^", ms=10, mfc="w", mec="k",
             transform=ccrs.PlateCarree(), zorder=7)
    cb = fig.colorbar(pm, ax=axm, pad=0.02, shrink=0.8); cb.set_label(r"$Z_H$ (dBZ)")
    axm.set_title("KLOT 0.5 deg reflectivity", loc="left")

    # --- stitched QVP with moving cursor ---
    for r, (key, cmap, vmn, vmx, cbl) in enumerate(qpanels):
        ax = fig.add_subplot(gs[r, 1])
        pm = ax.pcolormesh(QV["xe"], QV["he"], QV[key].T, cmap=cmap, vmin=vmn, vmax=vmx,
                           shading="flat", rasterized=True)
        cb = fig.colorbar(pm, ax=ax, pad=0.012, fraction=0.046); cb.set_label(cbl)
        ax.set_ylim(0, 10); ax.set_ylabel("Ht (km)")
        ax.axhline(4.0, color="0.3", lw=0.8, ls="--", zorder=6)
        ax.axvline(fi + 0.5, color="k", lw=1.6, zorder=8)
        ax.set_xlim(0, QV["nX"]); ax.set_xticks(QV["tick_i"] + 0.5)
        ax.set_xticklabels(QV["tick_lab"] if r == 2 else [])
        if r == 0: ax.set_title("Quasi-vertical profile (4.0 deg tilt)", loc="left", fontsize=9)
        if r == 2: ax.set_xlabel("Time (UTC)")
    fig.suptitle("KLOT convective event 2022-08-29/30 — %s UTC  ·  frame %d/%d"
                 % (pd.Timestamp(t).strftime("%Y-%m-%d %H:%M:%S"), fi + 1, QV["nX"]),
                 x=0.04, ha="left", fontsize=11)
    fig.savefig(path, dpi=100); plt.close(fig)

In [ ]:
import glob
from PIL import Image
os.makedirs("frames", exist_ok=True)

good, bad = [], []
for fi, t in enumerate(tv):
    try:
        render_frame(fi, t, "frames/cf_%02d.png" % fi)
        good.append(fi)
    except Exception as e:                # skip scans with corrupt time-coord units
        bad.append((fi, str(pd.Timestamp(t))))
    gc.collect()
print("rendered %d frames, skipped %d: %s" % (len(good), len(bad), [b[0] for b in bad]))

files = sorted(glob.glob("frames/cf_*.png"))
imgs  = [Image.open(f).convert("RGB") for f in files]
scale = 0.72
imgs  = [im.resize((int(im.width * scale), int(im.height * scale)), Image.LANCZOS)
         for im in imgs]
durs  = [420] * len(imgs); durs[-1] = 1400
imgs[0].save("KLOT_PPI_QVP_20220830.gif", save_all=True, append_images=imgs[1:],
             duration=durs, loop=0, optimize=True)
print("wrote KLOT_PPI_QVP_20220830.gif  (%.1f MB, %d frames)"
      % (os.path.getsize("KLOT_PPI_QVP_20220830.gif") / 1e6, len(imgs)))

## Result

![KLOT PPI + stitched QVP animation](KLOT_QVP_outputs/KLOT_PPI_QVP_20220830.gif)

The physics reads cleanly across the loop: deep warm-rain Z_DR (> 3 dB) below ~3.5 km, ρHV ≈
0.99 through the echo, and convective cells building and passing through the domain
00:00–01:30 UTC. The dashed line marks the ~0 °C level (≈ 4 km) for this warm-season case.

---
*NEXRAD ARCO store: https://registry.opendata.aws/nexrad-arco/ · Built with Py-ART, xradar,
icechunk, cmweather. Authored by Scott Collis with assistance from Claude (Anthropic).*